# Machine Learning Model Deployment Assignment
## Model Development & Preparation

**Author:** Bikash Subedi  
**Course:** Capstone Project  
**Program:** Artificial Intelligence (AIGC)  
**Institution:** Seneca Polytechnic  
**Date:** June 12, 2026  

---
**Project Overview:** The objective of this project is to develop a predictive regression model for food delivery time estimation and successfully deploy it via a REST API to a cloud environment.

**Dataset:** Delivery Time Estimation Analysis (Source: Kaggle: https://www.kaggle.com/datasets/ithikashr/delivery-time-estimationanalysis)

In [1]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from fastapi import FastAPI, HTTPException, Security
from fastapi.security import APIKeyHeader
from pydantic import BaseModel
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException, Security
from fastapi.testclient import TestClient
from fastapi.security import APIKeyHeader
from pydantic import BaseModel
import joblib
import pandas as pd

# 1. Problem & Data Selection
This phase builds the core predictive engine that learns from historical delivery patterns. We gather data like delivery distances and driver ratings, feeding it into algorithms such as Random Forest to see how these variables affect delivery speed. The goal is a model that estimates delivery times with the lowest error possible.



In [2]:
# Dataset
df = pd.read_csv('DeliveryTimeTaken.csv')
df.sample(3)

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken
35288,0x774a,HYDRES14DEL02,36.0,4.7,17.426228,78.407495,17.556228,78.537495,14-03-2022,23:25,23:40,conditions,Low,0,Buffet,motorcycle,1.0,No,Metropolitian,23.0
5099,0x44db,VADRES20DEL03,35.0,4.9,22.311358,73.164798,22.341358,73.194798,09-03-2022,20:40,20:55,conditions,Jam,1,Buffet,motorcycle,1.0,No,Metropolitian,27.0
13949,0x4e15,JAPRES20DEL01,39.0,4.1,26.956431,75.776649,26.996431,75.816649,05-03-2022,12:10,12:25,conditions,High,1,Meal,scooter,1.0,No,Metropolitian,34.0


# 2. Data Preprocessing
Machine learning algorithms need clean numerical input, but real-world data is often messy or text-based. To handle this, we build an automated pipeline that cleans missing values and converts categories (like weather conditions) into numerical format. The output is a single file containing both the preprocessing rules and the trained model, so the two stay in sync.



In [3]:
# Data Clening
# Haversine formula to calculate distance 
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1)/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2.0)**2
    return 6371 * (2 * np.arcsin(np.sqrt(a)))

df['Distance_km'] = haversine_distance(
    df['Restaurant_latitude'], df['Restaurant_longitude'], 
    df['Delivery_location_latitude'], df['Delivery_location_longitude']
)

columns_to_drop = ['ID', 'Delivery_person_ID', 'Order_Date', 'Time_Orderd', 'Time_Order_picked',
                   'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']
df_cleaned = df.drop(columns=columns_to_drop)
df_cleaned

,Delivery_person_Age,Delivery_person_Ratings,Weather,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken,Distance_km
0,37.0,4.9,conditions,High,2,Snack,motorcycle,0.0,No,Urban,24.0,3.025149
1,34.0,4.5,conditions,Jam,2,Snack,scooter,1.0,No,Metropolitian,33.0,20.183530
2,23.0,4.4,conditions,Low,0,Drinks,motorcycle,1.0,No,Urban,26.0,1.552758
3,38.0,4.7,conditions,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,21.0,7.790401
4,32.0,4.6,conditions,High,1,Snack,scooter,1.0,No,Metropolitian,30.0,6.210138
...,...,...,...,...,...,...,...,...,...,...,...,...
45588,30.0,4.8,conditions,High,1,Meal,motorcycle,0.0,No,Metropolitian,32.0,1.489846
45589,21.0,4.6,conditions,Jam,0,Buffet,motorcycle,1.0,No,Metropolitian,36.0,11.007735
45590,30.0,4.9,conditions,Low,1,Drinks,scooter,0.0,No,Metropolitian,16.0,4.657195
45591,20.0,4.7,conditions,High,0,Snack,motorcycle,1.0,No,Metropolitian,26.0,6.232393


# 3. Model Training & Selection
This stage turns the static model into a live application. We wrap the saved model in a FastAPI server, secure it with an access key, and package everything in a Docker container for cloud hosting. The result is an endpoint that takes a JSON request with order details and returns an estimated delivery time in minutes.


In [4]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

best_model = None
best_score = float('inf')
best_model_name = ""

In [5]:
# Train and evaluate
df_cleaned['Weather'] = df_cleaned['Weather'].astype(str).str.replace('conditions', '', regex=False).str.strip()

X = df_cleaned.drop('Time_taken', axis=1)
y = df_cleaned['Time_taken']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition', 
                    'multiple_deliveries', 'Distance_km']
categorical_features = ['Weather', 'Road_traffic_density', 'Type_of_order', 
                        'Type_of_vehicle', 'Festival', 'City']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

In [6]:
best_score = float('inf')
best_model_name = ""
best_model = None

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"{name} - MAE: {mae:.2f}, R2: {r2:.2f}")
    
    if mae < best_score:
        best_score = mae
        best_model = pipeline
        best_model_name = name

print(f"\nSelected Model: {best_model_name}")
joblib.dump(best_model, 'delivery_model.joblib')
print("Model saved successfully.")

Linear Regression - MAE: 5.22, R2: 0.50
Random Forest - MAE: 4.48, R2: 0.63

Selected Model: Random Forest
Model saved successfully.


# API Endpoint Configuration:

In [7]:
# Initialize API and Security
app = FastAPI(title="Delivery Time Prediction API")
API_KEY = "seneca-aigc-secret-key"
api_key_header = APIKeyHeader(name="X-API-Key")

# Load model at startup
model = joblib.load('delivery_model.joblib')

# Define the expected JSON input structure
class DeliveryRequest(BaseModel):
    Delivery_person_Age: float
    Delivery_person_Ratings: float
    Vehicle_condition: int
    multiple_deliveries: float
    Distance_km: float
    Weather: str
    Road_traffic_density: str
    Type_of_order: str
    Type_of_vehicle: str
    Festival: str
    City: str

# Authentication function
def get_api_key(api_key: str = Security(api_key_header)):
    if api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return api_key

# Create the POST endpoint
@app.post("/predict")
def predict_delivery_time(request: DeliveryRequest, api_key: str = Security(get_api_key)):
    input_data = pd.DataFrame([request.model_dump()])
    prediction = model.predict(input_data)
    return {"estimated_time_minutes": round(prediction[0], 2)}

In [8]:
client = TestClient(app)

# The mock data we are sending to the API
test_data = {
    "Delivery_person_Age": 32.0,
    "Delivery_person_Ratings": 4.8,
    "Vehicle_condition": 1,
    "multiple_deliveries": 0.0,
    "Distance_km": 5.5,
    "Weather": "Cloudy",
    "Road_traffic_density": "Medium",
    "Type_of_order": "Meal",
    "Type_of_vehicle": "motorcycle",
    "Festival": "No",
    "City": "Urban"
}

# The required security header
headers = {
    "X-API-Key": "seneca-aigc-secret-key"
}

# Send the POST request
response = client.post("/predict", json=test_data, headers=headers)

print(f"Status Code: {response.status_code}")
print(f"API Response: {response.json()}")

Status Code: 200
API Response: {'estimated_time_minutes': 18.11}
